In [1]:
from bs4 import BeautifulSoup as soup
import requests

### Grabbing the URL, Set, and Packs from the main website

First, we find the set, and the correspond packs for those sets.

In [2]:
from bs4.element import Tag

main_page = "https://pkmncards.com/sets/"

html    = requests.get(main_page)
parsed  = soup(html.content)
sets    = parsed.find_all("h2")

sets_and_grouped_packs : dict[str, Tag] = {s.text:s.find_next_sibling() for s in sets}

Parsing out the `Tag` into individual packs

In [3]:
sets_and_split_packs : dict[str, list[tuple[str, str]]] = { }

# Parsing out the URL and name of each individual pack
for s, grouped_packs in sets_and_grouped_packs.items():
	raw_packs = grouped_packs.find_all("a")
	parsed_packs = []
	for p in raw_packs:
		sets_and_split_packs.setdefault(s, []).append((p.text, p.get("href")))

sets_and_split_packs

{'Mega Evolution': [('Chaos Rising (CRI)',
   'https://pkmncards.com/set/chaos-rising/'),
  ('Perfect Order (POR)', 'https://pkmncards.com/set/perfect-order/'),
  ('Ascended Heroes (ASC)', 'https://pkmncards.com/set/ascended-heroes/'),
  ('Phantasmal Flames (PFL)', 'https://pkmncards.com/set/phantasmal-flames/'),
  ('Mega Evolution Energy (MEE)', 'https://pkmncards.com/set/mee/'),
  ('Mega Evolution (MEG)', 'https://pkmncards.com/set/mega-evolution/'),
  ('Mega Evolution Promos (MEP)',
   'https://pkmncards.com/set/mega-evolution-promos/')],
 'Scarlet & Violet': [('White Flare (WHT)',
   'https://pkmncards.com/set/white-flare/'),
  ('Scarlet & Violet Energy (SVE)',
   'https://pkmncards.com/set/scarlet-violet-energy/'),
  ('Black Bolt (BLK)', 'https://pkmncards.com/set/black-bolt/'),
  ('Destined Rivals (DRI)', 'https://pkmncards.com/set/destined-rivals/'),
  ('Journey Together (JTG)', 'https://pkmncards.com/set/journey-together/'),
  ('Prismatic Evolutions (PRE)',
   'https://pkmncard

### Fetching card information for each pack

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

session = requests.Session()
MAX_WORKERS = 16

def parse_pack(s, pack_name, pack_url):
	url  = pack_url + "?sort=date&ord=auto&display=full"
	html = session.get(url).content
	parsed = soup(html, "lxml")

	pack_cards = []
	for c in parsed.find_all(class_="type-pkmn_card"):
		name = c.find("span", title="Name").text
		image_url = c.find("img", class_="card-image").get("src")

		price_box = c.find("div", class_="card-pricing available")
		if price_box:
			l = price_box.find("li", title="Lowest Price").span.text
			m = price_box.find("li", title="Market Price").span.text
			h = price_box.find("li", title="Highest Price").span.text
		else:
			l = m = h = None

		pack_cards.append({
			"set": s,
			"pack": pack_name,
			"name": name,
			"image_url": image_url,
			"lowest_price": l,
			"market_price": m,
			"highest_price": h
		})
	return pack_cards

jobs = [(s, pack_name, pack_url)
		for s, packs in sets_and_split_packs.items()
		for pack_name, pack_url in packs]

cards = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
	futures = [executor.submit(parse_pack, s, pack_name, pack_url) for s, pack_name, pack_url in jobs]
	for future in tqdm(as_completed(futures), total=len(futures), desc="Packs"):
		cards.extend(future.result())

cards_df = pd.DataFrame(cards)
cards_df

Packs:   0%|          | 0/240 [00:00<?, ?it/s]